# Week 6: Data Retrieval and Processing

**Project:** Smart Logistics Tracking System  
**Company:** Kaizen Logistics  
**Milestone Source:** Week 5 - Smart Tracking System Blockchain Ledger  
**Blockchain Environment:** Ganache Local Ethereum Testnet  
**Smart Contract:** `IoTDataStorage.sol`

This notebook retrieves the 100 package records stored on the blockchain, decodes the package data, cleans and structures the dataset, and exports the final processed CSV file for Week 7 visualization.

## 1. Homework Objective

The objective of this activity is to retrieve IoT package records from the blockchain ledger and prepare the data for analysis and visualization. The retrieved records are decoded from the smart contract output, converted into a structured DataFrame, cleaned for missing values and proper data types, and saved as `assets/cleaned_iot_data.csv`.

This process supports the next visualization stage by ensuring that dates, coordinates, temperature readings, delivery status, RFID metrics, and exception fields are ready for analysis.

## 2. Import Required Libraries

In [1]:
from pathlib import Path
from web3 import Web3

import json
import pandas as pd
import numpy as np

## 3. Connect Python to Ganache

Ganache must be running before this notebook is executed. The RPC URL below should match the Ganache server port shown in the Ganache application.

In [2]:
# Connect to the local Ganache blockchain
ganache_url = "http://127.0.0.1:7545"

web3 = Web3(Web3.HTTPProvider(ganache_url, request_kwargs={"timeout": 60}))

if web3.is_connected():
    print("Connected to Ganache successfully.")
    print("RPC URL:", ganache_url)
    print("Current block number:", web3.eth.block_number)
    print("Available Ganache accounts:", len(web3.eth.accounts))
else:
    raise ConnectionError("Connection failed. Ensure Ganache is running and the RPC URL is correct.")

Connected to Ganache successfully.
RPC URL: http://127.0.0.1:7545
Current block number: 101
Available Ganache accounts: 10


## 4. Load the Smart Contract

In [3]:
# Deployed smart contract details
contract_address = "0xf3E41020bA9AAf69a3B215649E23bEb719cf742E"
owner_address = "0xEd6FB6c23A9751893a8483d070e583e212a88E1A"

# Convert address to checksum format
contract_address = web3.to_checksum_address(contract_address)

# Locate the ABI file from the repository contracts folder
candidate_abi_paths = [
    Path("contracts") / "abi.json",
    Path("abi.json"),
    Path("../contracts") / "abi.json"
]

abi_path = None
for path in candidate_abi_paths:
    if path.exists():
        abi_path = path
        break

if abi_path is None:
    raise FileNotFoundError("abi.json was not found. Expected location: contracts/abi.json")

with open(abi_path, "r") as file:
    abi = json.load(file)

# Load the deployed smart contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Set default sender for possible calls or transactions
web3.eth.default_account = web3.eth.accounts[0]

print("ABI loaded successfully from:", abi_path)
print("Connected to Smart Contract at:", contract_address)
print("Default sender account:", web3.eth.default_account)

if web3.eth.default_account.lower() != owner_address.lower():
    print("Warning: Default sender does not match the recorded owner/deployer address.")
else:
    print("Default sender matches the recorded owner/deployer address.")

ABI loaded successfully from: contracts/abi.json
Connected to Smart Contract at: 0xf3E41020bA9AAf69a3B215649E23bEb719cf742E
Default sender account: 0xEd6FB6c23A9751893a8483d070e583e212a88E1A
Default sender matches the recorded owner/deployer address.


## 5. Get the Total Number of Stored Records

In [4]:
# Retrieve the total number of records stored in the smart contract
total_records = contract.functions.getTotalRecords().call()

print(f"Total IoT records stored: {total_records}")

if total_records == 0:
    raise ValueError("No blockchain records were found. Complete the Milestone 1 upload before running Week 6.")

if total_records != 100:
    print("Warning: Expected 100 package records. Check whether the correct contract address is being used.")

Total IoT records stored: 100


## 6. Retrieve All Stored Blockchain Records

Each smart contract record contains four fields: blockchain timestamp, package ID, data type, and data value. The `data_value` field contains the full package row as a JSON string from the Week 5 blockchain ledger submission.

In [5]:
# Retrieve all IoT records from the blockchain
raw_records = []

for index in range(total_records):
    record = contract.functions.getRecord(index).call()

    raw_records.append({
        "blockchain_index": index,
        "blockchain_timestamp": record[0],
        "ledger_package_id": record[1],
        "ledger_data_type": record[2],
        "data_value": record[3]
    })

raw_ledger_df = pd.DataFrame(raw_records)

# Convert blockchain timestamp to readable datetime
raw_ledger_df["blockchain_datetime"] = pd.to_datetime(
    raw_ledger_df["blockchain_timestamp"],
    unit="s",
    errors="coerce"
)

print("Retrieved blockchain records:", len(raw_ledger_df))
raw_ledger_df.head()

Retrieved blockchain records: 100


,blockchain_index,blockchain_timestamp,ledger_package_id,ledger_data_type,data_value,blockchain_datetime
0,0,1781967685,PKG033,PackageRecord,"{""Current Latitude"": ""35.7214"", ""Current Locat...",2026-06-20 15:01:25
1,1,1781967686,PKG023,PackageRecord,"{""Current Latitude"": ""34.4398"", ""Current Locat...",2026-06-20 15:01:26
2,2,1781967686,PKG001,PackageRecord,"{""Current Latitude"": ""37.8648"", ""Current Locat...",2026-06-20 15:01:26
3,3,1781967686,PKG039,PackageRecord,"{""Current Latitude"": ""35.5355"", ""Current Locat...",2026-06-20 15:01:26
4,4,1781967686,PKG002,PackageRecord,"{""Current Latitude"": ""33.920485"", ""Current Loc...",2026-06-20 15:01:26


## 7. Decode JSON Package Records

The package record is stored inside `data_value` as a JSON string. This section decodes each JSON value back into individual package columns while preserving the original CSV column order.

In [6]:
# Decode JSON package rows from the blockchain data_value field
decoded_rows = []

for _, row in raw_ledger_df.iterrows():
    decoded_data = json.loads(row["data_value"])

    # Add blockchain metadata after the original package fields
    decoded_data["blockchain_index"] = row["blockchain_index"]
    decoded_data["blockchain_timestamp"] = row["blockchain_timestamp"]
    decoded_data["blockchain_datetime"] = row["blockchain_datetime"]
    decoded_data["ledger_package_id"] = row["ledger_package_id"]
    decoded_data["ledger_data_type"] = row["ledger_data_type"]

    decoded_rows.append(decoded_data)

df = pd.DataFrame(decoded_rows)

# Preserve the expected package column order from the Kaizen Logistics source CSV
expected_package_columns = [
    "package_id",
    "tracking_number",
    "timestamp",
    "Origin Location",
    "Origin City",
    "Origin Prefecture",
    "Origin Longitude",
    "Origin Latitude",
    "Order Date",
    "Current Location",
    "Estimated Delivery Date",
    "Delivery Exception Reason",
    "Status",
    "Perishable",
    "Temperature",
    "Temperature Issue",
    "Current Longitude",
    "Current Latitude",
    "Delivery Longitude",
    "Delivery Latitude",
    "Delivery City",
    "Delivery Prefecture",
    "Route Distance KM",
    "Estimated Transit Hours",
    "RFID #",
    "RFID Verified",
    "RFID Failure %",
    "RFID Failure Label",
    "RFID Success %",
    "RFID Success Label"
]

metadata_columns = [
    "blockchain_index",
    "blockchain_timestamp",
    "blockchain_datetime",
    "ledger_package_id",
    "ledger_data_type"
]

available_package_columns = [col for col in expected_package_columns if col in df.columns]
available_metadata_columns = [col for col in metadata_columns if col in df.columns]
df = df[available_package_columns + available_metadata_columns]

print("Decoded DataFrame shape:", df.shape)
print("Decoded columns:")
print(df.columns.tolist())

df.head()

Decoded DataFrame shape: (100, 35)
Decoded columns:
['package_id', 'tracking_number', 'timestamp', 'Origin Location', 'Origin City', 'Origin Prefecture', 'Origin Longitude', 'Origin Latitude', 'Order Date', 'Current Location', 'Estimated Delivery Date', 'Delivery Exception Reason', 'Status', 'Perishable', 'Temperature', 'Temperature Issue', 'Current Longitude', 'Current Latitude', 'Delivery Longitude', 'Delivery Latitude', 'Delivery City', 'Delivery Prefecture', 'Route Distance KM', 'Estimated Transit Hours', 'RFID #', 'RFID Verified', 'RFID Failure %', 'RFID Failure Label', 'RFID Success %', 'RFID Success Label', 'blockchain_index', 'blockchain_timestamp', 'blockchain_datetime', 'ledger_package_id', 'ledger_data_type']


,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,blockchain_index,blockchain_timestamp,blockchain_datetime,ledger_package_id,ledger_data_type
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,YES,1.98,Low Risk,98.02,Excellent,0,1781967685,2026-06-20 15:01:25,PKG033,PackageRecord
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,YES,0.91,Low Risk,99.09,Excellent,1,1781967686,2026-06-20 15:01:26,PKG023,PackageRecord
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,YES,2.66,Low Risk,97.34,Excellent,2,1781967686,2026-06-20 15:01:26,PKG001,PackageRecord
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,YES,0.92,Low Risk,99.08,Excellent,3,1781967686,2026-06-20 15:01:26,PKG039,PackageRecord
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,YES,2.65,Low Risk,97.35,Excellent,4,1781967686,2026-06-20 15:01:26,PKG002,PackageRecord


## 8. Clean Missing and Text Values

Delivered packages do not have delivery exceptions. For cleaner filtering and visualization, blank or missing values in `Delivery Exception Reason` are standardized as `None`. Text categories are also normalized for consistent downstream analysis.

In [7]:
# Standardize missing or blank delivery exception reasons
if "Delivery Exception Reason" in df.columns:
    df["Delivery Exception Reason"] = (
        df["Delivery Exception Reason"]
        .replace(["", "nan", "NaN", "None", None, np.nan], "None")
        .fillna("None")
    )

# Standardize common categorical text fields
categorical_columns = ["Status", "Perishable", "Temperature Issue", "RFID Verified", "RFID Failure Label", "RFID Success Label"]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Make package and tracking identifiers consistent text fields
identifier_columns = ["package_id", "tracking_number", "RFID #", "ledger_package_id", "ledger_data_type"]

for col in identifier_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("Missing value count after text cleanup:")
print(df.isna().sum())

df.head()

Missing value count after text cleanup:
package_id                   0
tracking_number              0
timestamp                    0
Origin Location              0
Origin City                  0
Origin Prefecture            0
Origin Longitude             0
Origin Latitude              0
Order Date                   0
Current Location             0
Estimated Delivery Date      0
Delivery Exception Reason    0
Status                       0
Perishable                   0
Temperature                  0
Temperature Issue            0
Current Longitude            0
Current Latitude             0
Delivery Longitude           0
Delivery Latitude            0
Delivery City                0
Delivery Prefecture          0
Route Distance KM            0
Estimated Transit Hours      0
RFID #                       0
RFID Verified                0
RFID Failure %               0
RFID Failure Label           0
RFID Success %               0
RFID Success Label           0
blockchain_index             0

,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,blockchain_index,blockchain_timestamp,blockchain_datetime,ledger_package_id,ledger_data_type
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,YES,1.98,Low Risk,98.02,Excellent,0,1781967685,2026-06-20 15:01:25,PKG033,PackageRecord
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,YES,0.91,Low Risk,99.09,Excellent,1,1781967686,2026-06-20 15:01:26,PKG023,PackageRecord
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,YES,2.66,Low Risk,97.34,Excellent,2,1781967686,2026-06-20 15:01:26,PKG001,PackageRecord
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,YES,0.92,Low Risk,99.08,Excellent,3,1781967686,2026-06-20 15:01:26,PKG039,PackageRecord
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,YES,2.65,Low Risk,97.35,Excellent,4,1781967686,2026-06-20 15:01:26,PKG002,PackageRecord


## 9. Convert Dates and Numeric Fields

This section converts timestamp fields into datetime format and converts coordinates, temperature, transit time, distance, and RFID percentages into numeric fields. These conversions make the dataset ready for charts, filters, maps, and statistical summaries.

In [8]:
# Convert date and timestamp fields
date_columns = [
    "timestamp",
    "Order Date",
    "Estimated Delivery Date",
    "blockchain_datetime"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Convert numerical fields
numeric_columns = [
    "Origin Longitude",
    "Origin Latitude",
    "Current Longitude",
    "Current Latitude",
    "Delivery Longitude",
    "Delivery Latitude",
    "Route Distance KM",
    "Estimated Transit Hours",
    "Temperature",
    "RFID Failure %",
    "RFID Success %",
    "blockchain_index",
    "blockchain_timestamp"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data types after conversion:")
print(df.dtypes)

df.head()

Data types after conversion:
package_id                           object
tracking_number                      object
timestamp                    datetime64[ns]
Origin Location                      object
Origin City                          object
Origin Prefecture                    object
Origin Longitude                    float64
Origin Latitude                     float64
Order Date                   datetime64[ns]
Current Location                     object
Estimated Delivery Date      datetime64[ns]
Delivery Exception Reason            object
Status                               object
Perishable                           object
Temperature                         float64
Temperature Issue                    object
Current Longitude                   float64
Current Latitude                    float64
Delivery Longitude                  float64
Delivery Latitude                   float64
Delivery City                        object
Delivery Prefecture                  object
Rou

,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,blockchain_index,blockchain_timestamp,blockchain_datetime,ledger_package_id,ledger_data_type
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,YES,1.98,Low Risk,98.02,Excellent,0,1781967685,2026-06-20 15:01:25,PKG033,PackageRecord
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,YES,0.91,Low Risk,99.09,Excellent,1,1781967686,2026-06-20 15:01:26,PKG023,PackageRecord
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,YES,2.66,Low Risk,97.34,Excellent,2,1781967686,2026-06-20 15:01:26,PKG001,PackageRecord
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,YES,0.92,Low Risk,99.08,Excellent,3,1781967686,2026-06-20 15:01:26,PKG039,PackageRecord
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,YES,2.65,Low Risk,97.35,Excellent,4,1781967686,2026-06-20 15:01:26,PKG002,PackageRecord


## 10. Add Visualization-Ready Helper Columns

The helper columns below support Week 7 visualization. They do not replace the original fields; they add cleaner analysis-ready values such as order week, delivery duration, and delivery performance category.

In [9]:
# Add order week label for later dashboard filtering
if "Order Date" in df.columns:
    df["Order Week"] = np.where(
        df["Order Date"].dt.date.between(pd.to_datetime("2026-05-03").date(), pd.to_datetime("2026-05-09").date()),
        "May 3-9, 2026",
        np.where(
            df["Order Date"].dt.date.between(pd.to_datetime("2026-05-10").date(), pd.to_datetime("2026-05-16").date()),
            "May 10-16, 2026",
            "Other"
        )
    )

# Calculate estimated delivery duration in hours from order date to estimated delivery date
if "Order Date" in df.columns and "Estimated Delivery Date" in df.columns:
    df["Estimated Delivery Duration Hours"] = (
        (df["Estimated Delivery Date"] - df["Order Date"]).dt.total_seconds() / 3600
    ).round(2)

# Create delivery performance category
if "Status" in df.columns:
    df["Delivery Performance"] = np.where(
        df["Status"].eq("Delivered"),
        "Delivered",
        np.where(df["Status"].eq("In Transit"), "In Transit", "Exception")
    )

# Create RFID reliability category using RFID Success %
if "RFID Success %" in df.columns:
    df["RFID Reliability Category"] = pd.cut(
        df["RFID Success %"],
        bins=[-np.inf, 90, 97, np.inf],
        labels=["At Risk", "Good", "Excellent"]
    ).astype(str)

# Remove blockchain and ledger-only columns from the final cleaned dataset.
# These columns are useful for audit/retrieval validation, but they are not needed for Week 7 visualization.
blockchain_ledger_columns = [
    "blockchain_index",
    "blockchain_timestamp",
    "blockchain_datetime",
    "ledger_package_id",
    "ledger_data_type"
]

df = df.drop(columns=[col for col in blockchain_ledger_columns if col in df.columns])

print("Visualization helper columns added.")
print("Blockchain and ledger-only columns removed from the final cleaned dataset.")
df.head()

Visualization helper columns added.
Blockchain and ledger-only columns removed from the final cleaned dataset.


,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID #,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,Order Week,Estimated Delivery Duration Hours,Delivery Performance,RFID Reliability Category
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,RFID-KZ-0033,YES,1.98,Low Risk,98.02,Excellent,"May 3-9, 2026",33.54,Delivered,Excellent
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,RFID-KZ-0023,YES,0.91,Low Risk,99.09,Excellent,"May 3-9, 2026",39.08,Delivered,Excellent
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,RFID-KZ-0001,YES,2.66,Low Risk,97.34,Excellent,"May 3-9, 2026",27.88,Delivered,Excellent
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,RFID-KZ-0039,YES,0.92,Low Risk,99.08,Excellent,"May 3-9, 2026",50.35,Delivered,Excellent
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,RFID-KZ-0002,YES,2.65,Low Risk,97.35,Excellent,"May 3-9, 2026",45.18,In Transit,Excellent


## 11. Format Decimal Values for Consistency

Numeric values used for visualization are formatted consistently to two decimal places, except for longitude and latitude fields which are kept at six decimal places for mapping accuracy. Date and timestamp fields are preserved as datetime values.

In [10]:
# Keep coordinates precise for mapping
coordinate_columns = [
    "Origin Longitude",
    "Origin Latitude",
    "Current Longitude",
    "Current Latitude",
    "Delivery Longitude",
    "Delivery Latitude"
]

for col in coordinate_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").round(6)

# Format analysis-oriented decimal columns to two decimal places
two_decimal_columns = [
    "Temperature",
    "Route Distance KM",
    "Estimated Transit Hours",
    "RFID Failure %",
    "RFID Success %",
    "Estimated Delivery Duration Hours"
]

for col in two_decimal_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

print("Decimal formatting completed.")
print("Coordinate columns retained at 6 decimal places.")
print("Analysis numeric columns rounded to 2 decimal places.")

df.head()

Decimal formatting completed.
Coordinate columns retained at 6 decimal places.
Analysis numeric columns rounded to 2 decimal places.


,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID #,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,Order Week,Estimated Delivery Duration Hours,Delivery Performance,RFID Reliability Category
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,RFID-KZ-0033,YES,1.98,Low Risk,98.02,Excellent,"May 3-9, 2026",33.54,Delivered,Excellent
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,RFID-KZ-0023,YES,0.91,Low Risk,99.09,Excellent,"May 3-9, 2026",39.08,Delivered,Excellent
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,RFID-KZ-0001,YES,2.66,Low Risk,97.34,Excellent,"May 3-9, 2026",27.88,Delivered,Excellent
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,RFID-KZ-0039,YES,0.92,Low Risk,99.08,Excellent,"May 3-9, 2026",50.35,Delivered,Excellent
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,RFID-KZ-0002,YES,2.65,Low Risk,97.35,Excellent,"May 3-9, 2026",45.18,In Transit,Excellent


## 12. Validate Cleaned Dataset

In [11]:
# Validate row count and key fields before export
print("Cleaned dataset shape:", df.shape)

if len(df) != 100:
    print("Warning: Cleaned dataset does not contain exactly 100 records.")

if "package_id" in df.columns:
    print("Unique package IDs:", df["package_id"].nunique())

if "Status" in df.columns:
    print("\nStatus distribution:")
    print(df["Status"].value_counts())

if "Temperature Issue" in df.columns:
    print("\nTemperature issue distribution:")
    print(df["Temperature Issue"].value_counts())

if "Order Week" in df.columns:
    print("\nOrder week distribution:")
    print(df["Order Week"].value_counts())

if "RFID Success %" in df.columns:
    print("\nAverage RFID Success %:", round(df["RFID Success %"].mean(), 2))

# Confirm that blockchain and ledger-only columns are no longer in the final visualization dataset
removed_columns_check = [
    "blockchain_index",
    "blockchain_timestamp",
    "blockchain_datetime",
    "ledger_package_id",
    "ledger_data_type"
]

remaining_ledger_columns = [col for col in removed_columns_check if col in df.columns]
print("\nRemaining blockchain/ledger-only columns:", remaining_ledger_columns)

print("\nRemaining missing values:")
print(df.isna().sum())

df.head()

Cleaned dataset shape: (100, 34)
Unique package IDs: 100

Status distribution:
Status
Delivered        96
In Transit        2
Not Delivered     2
Name: count, dtype: int64

Temperature issue distribution:
Temperature Issue
Ambient        72
Cool           26
Danger Zone     2
Name: count, dtype: int64

Order week distribution:
Order Week
May 3-9, 2026      50
May 10-16, 2026    50
Name: count, dtype: int64

Average RFID Success %: 98.14

Remaining blockchain/ledger-only columns: []

Remaining missing values:
package_id                           0
tracking_number                      0
timestamp                            0
Origin Location                      0
Origin City                          0
Origin Prefecture                    0
Origin Longitude                     0
Origin Latitude                      0
Order Date                           0
Current Location                     0
Estimated Delivery Date              0
Delivery Exception Reason            0
Status            

,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID #,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,Order Week,Estimated Delivery Duration Hours,Delivery Performance,RFID Reliability Category
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,RFID-KZ-0033,YES,1.98,Low Risk,98.02,Excellent,"May 3-9, 2026",33.54,Delivered,Excellent
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,RFID-KZ-0023,YES,0.91,Low Risk,99.09,Excellent,"May 3-9, 2026",39.08,Delivered,Excellent
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,RFID-KZ-0001,YES,2.66,Low Risk,97.34,Excellent,"May 3-9, 2026",27.88,Delivered,Excellent
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,RFID-KZ-0039,YES,0.92,Low Risk,99.08,Excellent,"May 3-9, 2026",50.35,Delivered,Excellent
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,RFID-KZ-0002,YES,2.65,Low Risk,97.35,Excellent,"May 3-9, 2026",45.18,In Transit,Excellent


## 13. Save the Cleaned IoT Dataset

The cleaned dataset is saved to the repository `assets` folder as `cleaned_iot_data.csv`. This file will be used as the prepared input for Week 7 visualization.

In [12]:
# Ensure the assets directory exists
output_dir = Path("assets")
output_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned IoT data to a CSV file in the assets directory
output_path = output_dir / "cleaned_iot_data.csv"

# Prepare export copy so selected decimal values consistently display with two digits in the CSV.
export_df = df.copy()

two_decimal_columns = [
    "Temperature",
    "Route Distance KM",
    "Estimated Transit Hours",
    "RFID Failure %",
    "RFID Success %",
    "Estimated Delivery Duration Hours"
]

for col in two_decimal_columns:
    if col in export_df.columns:
        export_df[col] = pd.to_numeric(export_df[col], errors="coerce").map(
            lambda value: "" if pd.isna(value) else f"{value:.2f}"
        )

# Keep longitude and latitude precise for geolocation mapping.
coordinate_columns = [
    "Origin Longitude",
    "Origin Latitude",
    "Current Longitude",
    "Current Latitude",
    "Delivery Longitude",
    "Delivery Latitude"
]

for col in coordinate_columns:
    if col in export_df.columns:
        export_df[col] = pd.to_numeric(export_df[col], errors="coerce").map(
            lambda value: "" if pd.isna(value) else f"{value:.6f}"
        )

export_df.to_csv(output_path, index=False)

print(f"Cleaned IoT data saved successfully as {output_path}")
print("Exported dataset shape:", export_df.shape)

# Optional preview of the final exported dataset
export_df.head()

Cleaned IoT data saved successfully as assets/cleaned_iot_data.csv
Exported dataset shape: (100, 34)


,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,RFID #,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label,Order Week,Estimated Delivery Duration Hours,Delivery Performance,RFID Reliability Category
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.110900,35.727200,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,RFID-KZ-0033,YES,1.98,Low Risk,98.02,Excellent,"May 3-9, 2026",33.54,Delivered,Excellent
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.363900,33.365600,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,RFID-KZ-0023,YES,0.91,Low Risk,99.09,Excellent,"May 3-9, 2026",39.08,Delivered,Excellent
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.031900,39.815600,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,RFID-KZ-0001,YES,2.66,Low Risk,97.34,Excellent,"May 3-9, 2026",27.88,Delivered,Excellent
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.195600,39.642800,2026-05-03 10:32:15.317110,"Hida, Gifu",...,RFID-KZ-0039,YES,0.92,Low Risk,99.08,Excellent,"May 3-9, 2026",50.35,Delivered,Excellent
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.416900,34.276600,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,RFID-KZ-0002,YES,2.65,Low Risk,97.35,Excellent,"May 3-9, 2026",45.18,In Transit,Excellent


## 14. Week 6 Summary

The notebook successfully retrieved all stored IoT package records from the Ganache blockchain through the deployed `IoTDataStorage` smart contract. The blockchain records were decoded from JSON, structured into a DataFrame, cleaned for missing values and correct data types, and enhanced with visualization-ready helper columns.

For the final visualization dataset, blockchain and ledger-only audit columns were removed because they are not needed for Week 7 charts or dashboard preparation. Numeric analysis fields were formatted to two decimal places, while longitude and latitude fields were preserved with higher precision for map-based visualization.

The final output, `assets/cleaned_iot_data.csv`, is ready for Week 7 line plots, dashboard preparation, and logistics tracking visualizations.